## E-Commerce Data Analysis Project

In [4]:
import pandas as pd
df=pd.read_csv("global_ecommerce_clean.csv")
# print(df)

## Convert OrderDate & ShipDate to datetime. in global file

In [5]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["ShipDate"] = pd.to_datetime(df["ShipDate"])
global_clean = pd.read_csv("global_ecommerce_clean.csv",parse_dates=["OrderDate","ShipDate"])

df["DeliveryDays"] = (df["ShipDate"]-df["OrderDate"]).dt.days
df["DeliveryDays"] = df["DeliveryDays"].fillna(0).astype(int)

## Handle missing values using fillna() or dropna().

In [ ]:
print(df.isnull().sum())
df["Sales"] = df["Sales"].fillna(0)
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].mean())
df["Profit"] = df["Profit"].fillna(0)
df["City"] = df["City"].fillna("Unknown")
df["State"] = df["State"].fillna("Unknown")
df["OrderDate"] = df["OrderDate"].fillna(method="ffill")  # forward fill
df["ShipDate"] = df["ShipDate"].fillna(method="bfill")   # backward fill
df = df.dropna()            # drops rows with any NaN
# df = df.dropna(subset=["Sales", "Profit"])  # drops only if key cols are NaN
print(df)

## Remove duplicates.

In [ ]:
print("Total duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["OrderID"])
df = df.drop_duplicates(subset=["CustomerID", "ProductID"])
df = df.drop_duplicates(keep="first")  # default, keeps first occurrence
df = df.drop_duplicates(keep="last")   # keeps last occurrence
df = df.drop_duplicates(keep=False)    # removes all duplicates
print(df)

## 🔹 1. Total Sales, Profit, Quantity by Region, Segment, Category

In [ ]:
region_summary = df.groupby("Region")[["Sales", "Profit", "Quantity"]].sum()
print("Region-wise Summary:\n", region_summary)
segment_summary = df.groupby("Segment")[["Sales", "Profit", "Quantity"]].sum()
print("\nSegment-wise Summary:\n", segment_summary)
category_summary = df.groupby("Category")[["Sales", "Profit", "Quantity"]].sum()
print("\nCategory-wise Summary:\n", category_summary)

## 2. Profit Margin (Profit / Sales)

In [ ]:
df["ProfitMargin"] = df["Profit"] / df["Sales"]
profit_margin_summary = df.groupby("Category")["ProfitMargin"].mean()
print("\nAverage Profit Margin by Category:\n", profit_margin_summary)


## 3. Top 10 Customers by Sales & Profit

In [ ]:
top_customers_sales = df.groupby("CustomerID")["Sales"].sum().sort_values(ascending=False).head(10)
top_customers_profit = df.groupby("CustomerID")["Profit"].sum().sort_values(ascending=False).head(10)
print("Top 10 Customers by Sales:\n", top_customers_sales)
print("\nTop 10 Customers by Profit:\n", top_customers_profit)

## 4. Monthly Sales Trend (using resample)

In [ ]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
monthly_sales = df.set_index("OrderDate").resample("M")["Sales"].sum()
print("Monthly Sales Trend:\n", monthly_sales)
import matplotlib.pyplot as plt

monthly_sales.plot(kind="bar", figsize=(10,5), title="Monthly Sales Trend")
plt.ylabel("Total Sales")
plt.show()

## (A) Cohort Analysis: Customer Retention by First Purchase Month

In [ ]:
df["OrderMonth"] = df["OrderDate"].dt.to_period("M")
first_purchase = df.groupby("CustomerID")["OrderMonth"].min()
df["CohortMonth"] = df["CustomerID"].map(first_purchase)
df["CohortIndex"] = (df["OrderMonth"] - df["CohortMonth"]).apply(lambda x: x.n)
cohort_data = df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index="CohortMonth", columns="CohortIndex", values="CustomerID")
print("Cohort Analysis (Customer Retention Table):\n", cohort_pivot)

## (B) Year-over-Year (YoY) Sales Growth

In [ ]:
df["Year"] = df["OrderDate"].dt.year
yearly_sales = df.groupby("Year")["Sales"].sum()

# Calculate Year-over-Year Growth %
yoy_growth = yearly_sales.pct_change() * 100

print("Yearly Sales:\n", yearly_sales)
print("\nYoY Growth (%):\n", yoy_growth)

## (C) Shipping Performance

In [ ]:
# Delivery days
df["DeliveryDays"] = (df["ShipDate"] - df["OrderDate"]).dt.days

# Average delivery time
avg_delivery = df["DeliveryDays"].mean()

# Count delayed shipments (e.g., more than 7 days)
delayed_orders = df[df["DeliveryDays"] > 7]

print("Average Delivery Days:", avg_delivery)
print("Number of Delayed Orders:", len(delayed_orders))
shipmode_delivery = df.groupby("ShipMode")["DeliveryDays"].mean()
print("Average Delivery Days by Ship Mode:\n", shipmode_delivery)

## (D) Detect Outliers in Profit using IQR

In [ ]:
Q1 = df["Profit"].quantile(0.25)
Q3 = df["Profit"].quantile(0.75)
IQR = Q3 - Q1

# Outlier condition
outliers = df[(df["Profit"] < (Q1 - 1.5 * IQR)) | (df["Profit"] > (Q3 + 1.5 * IQR))]

print("Number of Outliers in Profit:", len(outliers))